# Admin: Student Registration Notebook

This notebook is for workshop organizers to create/update student accounts in the gateway.

Scope:
- bulk register students via `/register`,
- verify each student via `/user/{email}`,
- optional delete via `/user/{email}`,
- save operation logs to `outputs/admin`.

Important:
- `/register` must be enabled on the server (`APP_ALLOW_REGISTRATION_ENDPOINT=true`).
- Caller must have admin access to the gateway.


## Step 1: Install Dependencies


In [1]:
%pip install -q requests pandas



[notice] A new release of pip is available: 25.2 -> 26.0.1
[notice] To update, run: pip install --upgrade pip
Note: you may need to restart the kernel to use updated packages.


## Step 2: Load Utilities and Resolve Project Root


In [2]:
import os
import sys
import json
from datetime import datetime, timezone
from pathlib import Path

import pandas as pd
import requests

PROJECT_ROOT = None
for candidate in [Path.cwd(), *Path.cwd().parents, Path('/content/ESMT_Workshop')]:
    if (candidate / 'notebooks').exists() and (candidate / 'data').exists():
        PROJECT_ROOT = candidate
        break

if PROJECT_ROOT is None:
    raise FileNotFoundError('Project root not found. Run this notebook from ESMT_Workshop repository.')

print('PROJECT_ROOT =', PROJECT_ROOT)


PROJECT_ROOT = /Users/alex/Projects/estm/ESMT_Workshop


## Step 3: Gateway Configuration


In [3]:
# Gateway URL from your current workshop environment.
SERVICE_URL = os.getenv(
    'WORKSHOP_API_BASE_URL',
    'https://gemini-workshop-gateway-395622257429.europe-west4.run.app',
).rstrip('/')

# Admin identity used by protected endpoints: /register and /user/{email}.
# The gateway accepts X-Admin-Email and can also accept Bearer auth.
ADMIN_EMAIL = os.getenv('WORKSHOP_ADMIN_EMAIL', 'btc.esmt.workshop@gmail.com').strip()
ADMIN_BEARER_TOKEN = os.getenv('WORKSHOP_ADMIN_BEARER_TOKEN', '').strip()

# Safety toggle:
# - True  -> build payload and show preview only
# - False -> execute real API calls
DRY_RUN = False

if not ADMIN_EMAIL and not ADMIN_BEARER_TOKEN:
    raise ValueError('Set WORKSHOP_ADMIN_EMAIL or WORKSHOP_ADMIN_BEARER_TOKEN for admin endpoints.')

print('SERVICE_URL =', SERVICE_URL)
print('ADMIN_EMAIL =', ADMIN_EMAIL)
print('ADMIN token provided =', bool(ADMIN_BEARER_TOKEN))
print('DRY_RUN =', DRY_RUN)


SERVICE_URL = https://gemini-workshop-gateway-395622257429.europe-west4.run.app
ADMIN_EMAIL = btc.esmt.workshop@gmail.com
ADMIN token provided = False
DRY_RUN = False


## Step 4: API Helper Functions


In [4]:
def _headers() -> dict:
    headers = {'Content-Type': 'application/json'}
    if ADMIN_EMAIL:
        headers['X-Admin-Email'] = ADMIN_EMAIL
    if ADMIN_BEARER_TOKEN:
        headers['Authorization'] = f'Bearer {ADMIN_BEARER_TOKEN}'
    return headers


def _response_body(resp: requests.Response):
    try:
        return resp.json()
    except Exception:
        return resp.text


def health_check() -> dict:
    url = f"{SERVICE_URL}/health"
    resp = requests.get(url, timeout=20)
    return {
        'status_code': resp.status_code,
        'body': _response_body(resp),
    }


def probe_admin_access() -> dict:
    """Non-destructive admin auth check for /register."""
    url = f"{SERVICE_URL}/register"
    payload = {'users': []}
    if DRY_RUN:
        return {
            'dry_run': True,
            'url': url,
            'payload_preview': payload,
        }

    resp = requests.post(url, headers=_headers(), json=payload, timeout=30)
    return {
        'dry_run': False,
        'status_code': resp.status_code,
        'body': _response_body(resp),
    }


def register_students(users: list[dict]) -> dict:
    url = f"{SERVICE_URL}/register"
    payload = {'users': users}
    if DRY_RUN:
        return {
            'dry_run': True,
            'url': url,
            'payload_preview': payload,
        }

    resp = requests.post(url, headers=_headers(), json=payload, timeout=40)
    return {
        'dry_run': False,
        'status_code': resp.status_code,
        'body': _response_body(resp),
    }


def get_student(email: str) -> dict:
    url = f"{SERVICE_URL}/user/{email}"
    resp = requests.get(url, headers=_headers(), timeout=20)
    return {
        'email': email,
        'status_code': resp.status_code,
        'body': _response_body(resp),
    }


def delete_student(email: str) -> dict:
    url = f"{SERVICE_URL}/user/{email}"
    if DRY_RUN:
        return {
            'dry_run': True,
            'email': email,
            'url': url,
        }

    resp = requests.delete(url, headers=_headers(), timeout=20)
    return {
        'dry_run': False,
        'email': email,
        'status_code': resp.status_code,
        'body': _response_body(resp),
    }


def run_full_registration_pipeline(users: list[dict]) -> tuple[dict, pd.DataFrame]:
    """Run register + verify in one call for organizer convenience."""
    register_result = register_students(users)

    verification_rows = []
    for email in [u['email'] for u in users]:
        response = get_student(email)
        row = {
            'email': email,
            'status_code': response['status_code'],
            'body': response['body'],
            'blocked': '',
            'requests_used': '',
            'request_limit': '',
            'concurrency_cap': '',
        }
        if isinstance(response['body'], dict):
            row['blocked'] = response['body'].get('blocked', '')
            row['requests_used'] = response['body'].get('requests_used', '')
            row['request_limit'] = response['body'].get('request_limit', '')
            row['concurrency_cap'] = response['body'].get('concurrency_cap', '')
        verification_rows.append(row)

    return register_result, pd.DataFrame(verification_rows)


## Step 5: Run Health and Admin Access Checks


In [5]:
health = health_check()
admin_probe = probe_admin_access()

print('Health:', health)
print('Admin probe:', admin_probe)

if health['status_code'] != 200:
    raise RuntimeError(f'Health check failed: {health}')

if admin_probe['status_code'] != 200:
    raise RuntimeError(
        'Admin auth failed. Check WORKSHOP_ADMIN_EMAIL or WORKSHOP_ADMIN_BEARER_TOKEN. '
        f'Probe response: {admin_probe}'
    )


Health: {'status_code': 200, 'body': {'status': 'ok'}}
Admin probe: {'dry_run': False, 'status_code': 200, 'body': {'registered': 0}}


## Step 6: Define Student Input (Inline or CSV)


In [6]:
# Option A: inline table (edit directly).
students_df = pd.DataFrame([
    {
        'email': 'student1@esmt.edu',
        'alias': 'Student 1',
        'request_limit': 15000,
        'concurrency_cap': 1,
    },
    {
        'email': 'student2@esmt.edu',
        'alias': 'Student 2',
        'request_limit': 15000,
        'concurrency_cap': 1,
    },
])

# Option B: load from CSV (uncomment and set path).
# csv_path = PROJECT_ROOT / 'data/input/student_emails.csv'
# students_df = pd.read_csv(csv_path)

students_df


,email,alias,request_limit,concurrency_cap
0,student1@esmt.edu,Student 1,15000,1
1,student2@esmt.edu,Student 2,15000,1


## Step 7: Validate and Build `/register` Payload


In [7]:
required_cols = ['email']
missing = [c for c in required_cols if c not in students_df.columns]
if missing:
    raise ValueError(f'Missing required columns: {missing}')

# Normalize values and build API payload.
def _optional_int(value):
    if pd.isna(value) or str(value).strip() == '':
        return None
    return int(value)

users_payload = []
for _, row in students_df.fillna('').iterrows():
    item = {
        'email': str(row['email']).strip(),
    }

    alias = str(row.get('alias', '')).strip()
    if alias:
        item['alias'] = alias

    req_limit = _optional_int(row.get('request_limit', ''))
    if req_limit is not None:
        item['request_limit'] = req_limit

    conc_cap = _optional_int(row.get('concurrency_cap', ''))
    if conc_cap is not None:
        item['concurrency_cap'] = conc_cap

    users_payload.append(item)

print('Prepared users:', len(users_payload))
users_payload[:3]


Prepared users: 2


[{'email': 'student1@esmt.edu',
  'alias': 'Student 1',
  'request_limit': 15000,
  'concurrency_cap': 1},
 {'email': 'student2@esmt.edu',
  'alias': 'Student 2',
  'request_limit': 15000,
  'concurrency_cap': 1}]

## Step 8: Run Full Registration Pipeline


In [13]:
register_result, verification_df = run_full_registration_pipeline(users_payload)
register_result


{'dry_run': False, 'status_code': 200, 'body': {'registered': 2}}

## Step 9: Verify Users with `/user/{email}`


In [14]:
verification_df


,email,status_code,body,blocked,requests_used,request_limit,concurrency_cap
0,student1@esmt.edu,200,"{'email': 'student1@esmt.edu', 'alias': 'Stude...",False,0,15000,1
1,student2@esmt.edu,200,"{'email': 'student2@esmt.edu', 'alias': 'Stude...",False,0,15000,1


## Step 10: Save Registration Log Artifacts


In [10]:
save_dir = PROJECT_ROOT / 'outputs' / 'admin'
save_dir.mkdir(parents=True, exist_ok=True)

run_id = datetime.now(timezone.utc).strftime('%Y%m%dT%H%M%S%fZ')

input_path = save_dir / f'registration_input_{run_id}.csv'
verify_path = save_dir / f'registration_verification_{run_id}.csv'
meta_path = save_dir / f'registration_meta_{run_id}.json'

students_df.to_csv(input_path, index=False)
verification_df.to_csv(verify_path, index=False)

meta = {
    'run_id': run_id,
    'service_url': SERVICE_URL,
    'admin_email': ADMIN_EMAIL,
    'dry_run': DRY_RUN,
    'register_result': register_result,
}
meta_path.write_text(json.dumps(meta, ensure_ascii=True, indent=2) + '')

print('Saved input:', input_path)
print('Saved verification:', verify_path)
print('Saved meta:', meta_path)


Saved input: /Users/alex/Projects/estm/ESMT_Workshop/outputs/admin/registration_input_20260207T144011094311Z.csv
Saved verification: /Users/alex/Projects/estm/ESMT_Workshop/outputs/admin/registration_verification_20260207T144011094311Z.csv
Saved meta: /Users/alex/Projects/estm/ESMT_Workshop/outputs/admin/registration_meta_20260207T144011094311Z.json


## Step 11 (Optional): Delete Students


In [11]:
# Safety switch for delete operation.
ENABLE_DELETE = False

# By default uses the same emails as registration payload.
emails_to_delete = [u['email'] for u in users_payload]

if ENABLE_DELETE:
    delete_results = [delete_student(email) for email in emails_to_delete]
    pd.DataFrame(delete_results)
else:
    print('Deletion disabled. Set ENABLE_DELETE=True to execute deletes.')


Deletion disabled. Set ENABLE_DELETE=True to execute deletes.
